# 🦾 Introduction to Tool Calling with LangChain 🦜🔗

In this notebook you'll learn how to use LLM APIs for Tool Callling through LangChain. By the end of this notebook, you will know how to implement function calling, and leverage LangChain for advanced interactions.

## ⚙️ Setup

👉 Run the cell below to import a couple of basic libraries.

In [ ]:
%load_ext autoreload
%autoreload 2
from pprint import pprint
import pandas as pd
from IPython.display import Markdown

For this challenge you'll need your API key again.

👉 Run the cell below to load it.

In [ ]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

## 😶‍🌫️ Understanding Function Calling

Function calling allows LLMs to execute specific functions during their interactions. This is particularly useful for tasks like data retrieval, calculations, or triggering workflows.

In the LangChain documentation you will see the term Tool Calling instead.

### Why Use Function Calling?

- **Efficiency**: Offload complex operations to predefined functions.
- **Customization**: Tailor the LLM's behavior to your specific use case.
- **Integration**: Seamlessly connect the LLM with external systems or APIs.

### What does it do, and what doesn't it do?

It's important to know that "Function Calling" or "Function Calling" doesn't actually run the functions it calls. That is still up to you! It will only format the LLM's output as if a function was called.

Tool calling formats LLM outputs into predefined structures (e.g. JSON, dictionnaries) and ensures responses follow specified schemas with required fields.

You can use it to take a user's natural language request and turn it into structured data you can use to call functions.

## 🪄 Creating a function

### 🧑‍🍳 Context

Imagine you are creating an app for your users to find recipes.

- You don't have recipes yourself, but you have found this amazing website [recipes.lewagon.com](https://recipes.lewagon.com) with recipes.
- You realize you can scrape the recipes' information from that site.
- You want your end users to be able to query recipes using natural language.
- Based on the user's query you will filter the recipes on:
   - Main ingredient
   - Minimum and maximum preparation time
   - Difficulty level

In `recipe.py` you will find all the code to scrape the website. It's a more advanced version of the scraping challenge earlier in the bootcamp. The main function is the last one `get_recipes()` which takes a couple of parameters to filter the recipes.

There is no need to go into the code and understand everything. You can consider it as a black box: you know which arguments you can provide, and you know it returns a DataFrame. 

👉 Run the cell below to see it in action.

In [ ]:
import recipe
recipe.get_recipes("chocolate", difficulty_levels=["Easy", "Moderate"], max_prep_time=60, min_prep_time=30, max_pages=2)

🎯 Our task now: take the user's natural language query, convert it into the right arguments that we can use to call our function.

## 🧰 Implementing Function Calling with Gemini

In this section, we will walk through implementing function calling using Gemini through LangChain. This includes defining functions and passing them to the API.

LangChain has some [documentation](https://docs.langchain.com/oss/python/langchain/tools) on the concept and how to implement it. Don't hesitate to refer back to it if something is unclear.

### Converting a Python function into a `tool`

👉  To start, run the cell below to import `tool`.

In [ ]:
from langchain.tools import tool

👉 Have a look at the cell below, and the `get_recipes_tool()` function we define here.

You will notice that:
- It has a short docstring.
- It uses type hinting to very clearly define what parameters the function takes.
- It's implementation is very basic: it just calls the `get_recipes()` function from `recipes.py`.

Whatever magic you will see happening later is completely based on the first two points: the docstring and the type hinting. The implementation (the actual code) will be considered as a black box.

In [ ]:
def get_recipes_tool(ingredient: str,
                     max_pages: int = 3,
                     difficulty_levels: list[str] = [],
                     max_prep_time: int = 0,
                     min_prep_time: int = 0) -> pd.DataFrame:
    """"Scrape recipes from the internet for a given ingredient.

    This function scrapes recipes from the internet for a given ingredient.
    It returns a DataFrame containing the recipes, including their names,
    difficulty levels, and preparation times. The function allows filtering
    by difficulty levels and preparation times.
    """
    return recipe.get_recipes(ingredient, max_pages, difficulty_levels, max_prep_time, min_prep_time)

👉 Run this cell, just to verify that everything works:

In [ ]:
get_recipes_tool("chocolate", difficulty_levels=["Easy", "Moderate"], max_prep_time=60, min_prep_time=30, max_pages=2)

Run this to check that is just a classic Python function:

In [ ]:
print("Type:", type(get_recipes_tool))
print("A function does not have attributes:", get_recipes_tool.__dict__)

All good, now let's transform this into a LangChain `tool`. We need to do this so our LLM will know what it's supposed to output: the arguments for our function.

How to transform a function into a tool? We will use a so-called decorator.

**A decorator?** A Python decorator is a function that takes another function (or method) as input and returns a new function that usually extends or modifies the behavior of the original.

A decorator is applied by using `@decorator_name` directly above the function definition. By doing that, it will extend the behaviour of a function. In our case, it will transform our classic Python function into a LangChain `tool`.

👉 Let's do it:
- Go back to the cell where we defined `get_recipes_tool()`.
- Insert a new line directly above the `def get_recipes_tool(...)` line.
- Write `@tool` on that line.
- Execute the cell again.

That's it!

👉 Now check the type of `get_recipes_tool` again!

In [ ]:
@tool
def get_recipes_tool(ingredient: str,
                     max_pages: int = 3,
                     difficulty_levels: list[str] = [],
                     max_prep_time: int = 0,
                     min_prep_time: int = 0) -> pd.DataFrame:
    """"Scrape recipes from the internet for a given ingredient.

    This function scrapes recipes from the internet for a given ingredient.
    It returns a DataFrame containing the recipes, including their names,
    difficulty levels, and preparation times. The function allows filtering
    by difficulty levels and preparation times.
    """
    return recipe.get_recipes(ingredient, max_pages, difficulty_levels, max_prep_time, min_prep_time)

In [ ]:
type(get_recipes_tool)

👉 Run this cell to check the attributes of the tool:

In [ ]:
get_recipes_tool.__dict__.keys()

👉 Take the time to explore some of these attributes:

In [ ]:
print(get_recipes_tool.name)

In [ ]:
print(get_recipes_tool.description)

In [ ]:
get_recipes_tool.args_schema

The attribute we'll often come back to is this one:

In [ ]:
# This will generate a long output, make sure to click on 'scrollable element'
# at the bottom of the output to see the full content
pprint(get_recipes_tool.args_schema.model_json_schema())

What are we looking at?

This is a JSON schema, with the description of our function (tool) and of what arguments our function expects. This is what we'll pass on to our model as part of its prompt. That way the model will know what it should return us so we can use it to call our function.

How did LangChain get this information? It extracted it from the type hints we provided in the function definition.

### Using our model for function calling

Now that we have wrapped our function as a LangChain tool, we can use it together with a model.

Remember: we'll feed the model with a natural language query, and it should give us the arguments we can use to call our function (tool).

👉 Let's use Gemini again, again using LangChain's generic Chat Models.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

👉 Set the model's temperature to 0. Think about why we want to do this for this use case?

In [ ]:
model.temperature = 0

We now need to link our tool `get_recipes_tool` to our model. Check the [LangChain documentation](https://docs.langchain.com/oss/python/langchain/models#tool-calling) to find out how to do that.

👉 Link `get_recipes_tool` to the model:

In [ ]:
model_with_tools = model.bind_tools([get_recipes_tool])

We can now use our enhanced `model_with_tools` to translate a user query into a tool call.

👉 Inspect the response. How is it different from a regular API call?

In [ ]:
query = "Give me some recipes with chocolate that take less than an hour."
response = model_with_tools.invoke(query)
response

If all goes well, it tells us which functions to call, and which arguments to use for each tool:

In [ ]:
# The function to call and the arguments to pass to it
response.tool_calls

Note that this is a list. In our case we only gave it one tool, so it only called that one. If we provided it more tools, it could have given us one or more tools to use, depending on the user's query.

It understood our question. And it also translated "one hour" into the value `60`, because we want it in minutes. Cool.

👉 Now, let's use the response to call our function:

- Extract the arguments from the response (as a dict)
- Use `get_recipes_tool` with those arguments:
    - you can use your `get_recipes_tool`'s `.invoke()` method and pass it the arguments (still as a dict)

In [ ]:
args = response.tool_calls[0]['args']
get_recipes_tool.invoke(args).head()

Let's experiment further with some more precise queries.

👉 Start with the first one. Later on you can also try the other ones.

In [ ]:
query = "What are some not too difficult recipes with chocolate that take less than 60 minutes to prepare?"
# query = "What are some easy chocolate recipes with a preparation time of less than one hour?"
# query = "What are some not too hard chocolate recipes with a preparation time of less than an hour."
# query = "What are some chocolate recipes with a preparation time of less than an hour? I am an average cook."
# query = "What are some chocolate recipes with a preparation time of less than an hour? Don't make it difficult."
# query = "What are easy chocolate recipes with a preparation time of less than an hour?"
response = model_with_tools.invoke(query)
response


In [ ]:
args = response.tool_calls[0]['args']
print("Retrieved arguments:", args, '\n')
get_recipes_tool.invoke(args)

You should see that the tool is calling our scraping function, because you see the `Scraping page ...` messages.

There are two options now:
- With a bit of luck, it's returning recipes, but only easy ones. You would have expected some medium difficulty ones too, no? 
- It could also be that the tool doesn't return anything.

In both cases, try to understand why. Run the cells a couple of times and look at the arguments and the different outcomes. 

Compare the arguments the LLM returns us it to the result of our first tool call.

When in doubt, run the cell below to call our function immediately.

In [ ]:
recipe.get_recipes(ingredient="chocolate", max_pages=2, max_prep_time=60)

The values in the `difficulty` column are capitalized, and we have `Moderate` instead of `Medium`.

Our model doesn't know what exact values we are looking for. Simply because we didn't specify them.

We'll have to improve on that.

👉 Give the next query a try. It might fail: the model doesn't really understand what to do for this case. Let's try to do better.

In [ ]:
query = "What are some easy chocolate recipes with a preparation time of less than one hour?"
response = model_with_tools.invoke(query)
response

### Improving our tool

So far, LangChain extracted the information about our function mainly from the type hints. We gave it only a short docstring to work with.

But we can do even better. The docstring of our `get_recipes_tool()` function was very short. Let's use a longer one: in the cell below we redefine the function with a more extensive docstring. Nothing else changed.

In [ ]:
from typing import Literal

@tool
def get_recipes_tool(
    ingredient: str,
    max_pages: int = 1,
    difficulty_levels: list[str] = [],
    max_prep_time: int = 0,
    min_prep_time: int = 0
) -> pd.DataFrame:
    """
    Scrape recipes from the internet for a given ingredient.

    This function scrapes recipes from the internet for a given ingredient.
    It returns a DataFrame containing the recipes, including their names,
    difficulty levels, and preparation times. The function allows filtering
    by difficulty levels and preparation times.

    Args:
        ingredient: The ingredient to search for.
        max_pages: The number of pages to scrape. Default is 1.
        difficulty_levels: The difficulty levels to filter by.
            Default is an empty list, which means no filtering.
        max_prep_time: The maximum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.
        min_prep_time: The minimum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.

    Returns:
        A DataFrame containing the recipes for the given ingredient.
    """
    return recipe.get_recipes(ingredient, max_pages, difficulty_levels, max_prep_time, min_prep_time)

👉 Let's check the JSON schema again. Did it change anything?

In [ ]:
pprint(get_recipes_tool.args_schema.model_json_schema())

Not really. The value for `description` is much longer. But `properties` is still the same, despite all the extra information in the docstring.

👉 Anyway, let's give it a try:

In [ ]:
query = "What are some easy chocolate recipes with a preparation time of less than one hour?"
model_with_tools = model.bind_tools([get_recipes_tool])
response = model_with_tools.invoke(query)
response

Better! At least it knows what to do now.

We can also do better than just having that long docstring.

We can instruct LangChain to read the docstring, and parse the information from it and link it to the arguments we expect.

👉 To do so, replace the decorator by `@tool(parse_docstring=True)`, and execute the cell again. Check the `properties` again. LangChain has extracted even more information out of the docstring. That gives the model even more precise information to work with.

In [ ]:
from typing import Literal

@tool(parse_docstring=True)
def get_recipes_tool(
    ingredient: str,
    max_pages: int = 1,
    difficulty_levels: list[str] = [],
    max_prep_time: int = 0,
    min_prep_time: int = 0
) -> pd.DataFrame:
    """
    Scrape recipes from the internet for a given ingredient.

    This function scrapes recipes from the internet for a given ingredient.
    It returns a DataFrame containing the recipes, including their names,
    difficulty levels, and preparation times. The function allows filtering
    by difficulty levels and preparation times.

    Args:
        ingredient: The ingredient to search for.
        max_pages: The number of pages to scrape. Default is 1.
        difficulty_levels: The difficulty levels to filter by.
            Default is an empty list, which means no filtering.
        max_prep_time: The maximum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.
        min_prep_time: The minimum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.

    Returns:
        A DataFrame containing the recipes for the given ingredient.
    """
    return recipe.get_recipes(ingredient, max_pages, difficulty_levels, max_prep_time, min_prep_time)

👉 Check the schema again: you will see that the docstring is now divided over the description (which became much shorter), and all the parameter description are now tied to the properties.

In [ ]:
pprint(get_recipes_tool.args_schema.model_json_schema())

In [ ]:
query = "What are some easy chocolate recipes with a preparation time of less than one hour?"
model_with_tools = model.bind_tools([get_recipes_tool])
response = model_with_tools.invoke(query)
response

Despite that, our model still doesn't give the correct values for our difficulty levels.

Let's work on that.

### Enumerating acceptable values

Enumerating is fancy word for listing: we'll tell the model which values are acceptable. We'll do that by expanding our type hints a bit further. 

👉 Look at the cell below, and check how we now give a list of acceptable values for `difficulty_levels` in the type hint. We also described the acceptable values in the docstring.

In [ ]:
from typing import Literal

@tool(parse_docstring=True)
def get_recipes_tool(
    ingredient: str,
    max_pages: int = 1,
    difficulty_levels: list[Literal["Very Easy", "Easy", "Moderate", "Hard", "Very Hard"]] = [],
    max_prep_time: int = 0,
    min_prep_time: int = 0
) -> pd.DataFrame:
    """
    Scrape recipes from the internet for a given ingredient.

    This function scrapes recipes from the internet for a given ingredient.
    It returns a DataFrame containing the recipes, including their names,
    difficulty levels, and preparation times. The function allows filtering
    by difficulty levels and preparation times.

    Args:
        ingredient: The ingredient to search for.
        max_pages: The number of pages to scrape. Default is 1.
        difficulty_levels: The difficulty levels to filter by.
            Default is an empty list, which means no filtering.
        max_prep_time: The maximum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.
        min_prep_time: The minimum preparation time in minutes.
            Default is 0.
            If set to 0, no filtering will be applied.

    Returns:
        A DataFrame containing the recipes for the given ingredient.
    """
    return recipe.get_recipes(ingredient, max_pages, difficulty_levels, max_prep_time, min_prep_time)

👉 Check the schema again: you'll see that `difficulty_levels` now features a list of acceptable values!

In [ ]:
pprint(get_recipes_tool.args_schema.model_json_schema())

Our tool is much better described now. 

👉 Try it again with the queries from above. Does it perform better?

In [ ]:
query = "What are some not too difficult recipes with chocolate that take less than 60 minutes to prepare?"
# query = "What are some not too hard chocolate recipes with a preparation time of less than an hour."
# query = "What are some chocolate recipes with a preparation time of less than an hour? I am an average cook."
# query = "What are some chocolate recipes with a preparation time of less than an hour? Don't make it difficult."
# query = "What are easy chocolate recipes with a preparation time of less than an hour?"
model_with_tools = model.bind_tools([get_recipes_tool])
response = model_with_tools.invoke(query)
response


In [ ]:
args = response.tool_calls[0]['args']
print("Retrieved arguments:", args, '\n')
get_recipes_tool.invoke(args)

## 📜 Summary

In this notebook, we explored the integration of LangChain and Gemini for tool calling, focusing on creating and refining a recipe-scraping tool. Key highlights include:

- **Tool Creation**: We defined a `get_recipes_tool` function to scrape recipes based on user queries, leveraging type hints and docstrings for clarity.
- **Tool Enhancement**: Improved the tool by adding detailed docstrings, enumerating acceptable values, and enabling `parse_docstring` for better schema generation.
- **Model Integration**: Linked the tool with Gemini using LangChain, enabling the model to interpret natural language queries and generate structured arguments for the tool.
- **Function Calling**: Demonstrated how the model translates user queries into tool calls with appropriate arguments.
- **Validation and Debugging**: Iteratively refined the tool and queries to ensure accurate results, addressing issues like difficulty level mismatches.

This workflow showcases the power of combining LLMs with structured tools for advanced, context-aware interactions.

### Next Steps

- Experiment with defining your own functions and integrating them with Gemini.
- Explore how function calling can be used in real-world applications, such as data processing or workflow automation.

🏁 Congratulations! You now master basic prompting using LangChain, using multiple messages.

## [Optional] Implementation with Google's `genai` library

In [ ]:
from google import genai
from google.genai import types

# Define the function declaration for the model
get_recipes_function = {
    'name': 'get_recipes',
    'description': 'Scrape recipes from the internet for a given ingredient. This function scrapes recipes from the internet for a given ingredient. It returns a DataFrame containing the recipes, including their names, difficulty levels, and preparation times. The function allows filtering by difficulty levels and preparation times.',
    'parameters': {
        'type': 'object',
        'properties': {
            'ingredient': {
                'description': 'The ingredient to search for.',
                'title': 'Ingredient',
                'type': 'string'
            },
            'max_pages': {
                'default': 1,
                'description': 'The number of pages to scrape. Default is 1.',
                'title': 'Max Pages',
                'type': 'integer'
            },
            'difficulty_levels': {
                'default': [],
                'description': 'The difficulty levels to filter by. Default is an empty list, which means no filtering.',
                'items': {
                    'enum': ['Very Easy', 'Easy', 'Moderate', 'Hard', 'Very Hard'],
                    'type': 'string'
                },
                'title': 'Difficulty Levels',
                'type': 'array'
            },
            'max_prep_time': {
                'default': 0,
                'description': 'The maximum preparation time in minutes. Default is 0. If set to 0, no filtering will be applied.',
                'title': 'Max Prep Time',
                'type': 'integer'
            },
            'min_prep_time': {
                'default': 0,
                'description': 'The minimum preparation time in minutes. Default is 0. If set to 0, no filtering will be applied.',
                'title': 'Min Prep Time',
                'type': 'integer'
            }
        },
        'required': ['ingredient'],
    }
}

# Configure the client and tools
client = genai.Client()
tools = types.Tool(function_declarations=[get_recipes_function])
config = types.GenerateContentConfig(tools=[tools])

In [ ]:
# Define the query
query = "What are some chocolate recipes with a preparation time of less than one hour? I'm an average cook."

# Send request with function declarations
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=query,
    config=config,
)

# Check for a function call
if response.candidates[0].content.parts[0].function_call:
    function_call = response.candidates[0].content.parts[0].function_call
    print(f"Function to call: {function_call.name}")
    print(f"Arguments: {function_call.args}")
    display(recipe.get_recipes(**function_call.args))
else:
    print("No function call found in the response.")
    print(response.text)